## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value! This is the start of a lab that will last 2 days.

And we're going to hand-build an Agent Loop without any Agent Framework..

### First, some prep

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours! You should be able to download it from your LinkedIn profile; go to your profile page use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

import os
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# ============================================================
# FREE-first BOOTSTRAP — makes bare `OpenAI()` calls work WITHOUT an OpenAI key.
#
# Maps the best available FREE provider from your .env into
# OPENAI_API_KEY / OPENAI_BASE_URL env vars, so `OpenAI()` (no args)
# Just Works - every cell down the line benefits from this automatically.
#
# FREE priority:  Groq FREE > Gemini FREE > OpenRouter FREE > Ollama (local)
# ============================================================

_gr_key   = os.getenv("GROQ_API_KEY")
_gm_key   = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
_or_key   = os.getenv("OPENROUTER_API_KEY")
_oa_key   = os.getenv("OPENAI_API_KEY")

if _gr_key:
    _which = "Groq (FREE tier - fastest inference)"
    _key, _base = _gr_key, "https://api.groq.com/openai/v1"
elif _gm_key:
    _which = "Gemini (FREE tier - high quality)"
    _key, _base = _gm_key, "https://generativelanguage.googleapis.com/v1beta/openai/"
elif _or_key:
    _which = "OpenRouter (FREE models gateway)"
    _key, _base = _or_key, "https://openrouter.ai/api/v1"
elif _oa_key:
    _which = "OpenAI (paid tier)"
    _key, _base = _oa_key, None
else:
    _which = "Ollama (local - 100% FREE)"
    _key, _base = "ollama", "http://localhost:11434/v1"

os.environ["OPENAI_API_KEY"] = _key
if _base:
    os.environ["OPENAI_BASE_URL"] = _base
elif "OPENAI_BASE_URL" in os.environ:
    del os.environ["OPENAI_BASE_URL"]

# --- Build a robust `llm_call()` helper that:
#     1. picks the best FREE provider/model
#     2. falls across multiple models if a specific one 404s
#     3. falls across PROVIDERS if needed (Groq -> Gemini -> OR -> Ollama)
# Every cell in this lab can simply call  llm_call(messages)  and get a string answer.
# Or  llm_call(messages, want_response_object=True)  to get the raw ChatCompletion
# ---

from openai import OpenAI as _BOpenAI

# Pre-configure clients for each provider (only if key exists)
_GROQ   = _BOpenAI(api_key=_gr_key, base_url="https://api.groq.com/openai/v1") if _gr_key else None
_GEMINI = _BOpenAI(api_key=_gm_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/") if _gm_key else None
_OR     = _BOpenAI(api_key=_or_key, base_url="https://openrouter.ai/api/v1") if _or_key else None
_OLLAMA = _BOpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

# Preferred FREE models (confirmed valid - July/Aug 2026)
_MODEL_CHAINS = [
    # (client_var, [(model, label), ...])
    (_GROQ, [
        ("llama3-70b-8192",       "Groq FREE Llama 3 70B"),
        ("gemma2-9b-it",          "Groq FREE Gemma 2 9B"),
        ("llama-3.1-8b-instant",  "Groq FREE Llama 3.1 8B"),
        ("mixtral-8x7b-32768",    "Groq FREE Mixtral 8x7B"),
    ]),
    (_GEMINI, [
        ("gemini-2.5-flash",      "Gemini FREE 2.5 Flash"),
        ("gemini-2.0-flash",      "Gemini FREE 2.0 Flash"),
    ]),
    (_OR, [
        ("nvidia/nemotron-3-ultra-550b-a55b:free", "OR FREE Nemotron 3 Ultra"),
        ("poolside/laguna-s-2.1:free",             "OR FREE Laguna S 2.1"),
        ("deepseek/deepseek-r1:free",              "OR FREE DeepSeek R1"),
        ("meta-llama/llama-3.3-70b-instruct:free", "OR FREE Llama 3.3 70B"),
        ("openrouter/free",                        "OR FREE auto-router"),
    ]),
    (_OLLAMA, [
        ("llama3.2",    "Ollama llama3.2"),
        ("qwen2.5:3b",  "Ollama qwen2.5 3B"),
    ]),
]

_llm_used = None  # last (model_name, provider_label) used

def llm_call(messages, *, model=None, temperature=None, max_tokens=None,
             tools=None, tool_choice=None, want_response_object=False, verbose=True):
    """Universal FREE-first LLM call with cross-model/provider fallback.

    Returns the assistant's answer string by default, or the raw ChatCompletion
    object when want_response_object=True (needed for tool-call inspection).
    """
    global _llm_used

    # If a specific model was requested: try one specific (client, model, label).
    if model is not None:
        try_chain = [(_GROQ or _GEMINI or _OR or _OLLAMA, [(model, "Override: {}".format(model))])]
    else:
        try_chain = [(c, list(models)) for c, models in _MODEL_CHAINS if c is not None]

    last_err = None
    for client, models in try_chain:
        for model_name, label in models:
            kwargs = dict(model=model_name, messages=messages)
            if temperature is not None: kwargs["temperature"] = temperature
            if max_tokens is not None:    kwargs["max_tokens"] = max_tokens
            if tools is not None:         kwargs["tools"] = tools
            if tool_choice is not None:   kwargs["tool_choice"] = tool_choice
            try:
                if verbose:
                    print("  [LLM] {} (model={})".format(label, model_name))
                resp = client.chat.completions.create(**kwargs)
                _llm_used = (model_name, label)
                if want_response_object:
                    return resp
                choice_msg = resp.choices[0].message
                # If tool_calls: return the raw message object instead of content
                # (caller can inspect). But helper defaults to "answer string" when no tools.
                return choice_msg.content if choice_msg.content is not None else choice_msg
            except Exception as e:
                last_err = e
                if verbose:
                    print("  [LLM] failed: {}".format(e))
                continue

    raise RuntimeError(
        "llm_call failed on ALL providers/models. Last error: {}".format(last_err)
    )


# Also create the default `openai` client (backwards compatible for cells using it directly)
openai = OpenAI()

print("BOOTSTRAP OK - Default provider for bare OpenAI() calls: {}".format(_which))
if _base:
    print("BOOTSTRAP OK - Endpoint override: {}".format(_base))


BOOTSTRAP OK - Default provider for bare OpenAI() calls: Groq (FREE tier - fastest inference)
BOOTSTRAP OK - Endpoint override: https://api.groq.com/openai/v1


In [3]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
ed.donner@gmail.com
www.linkedin.com/in/eddonner
(LinkedIn)
edwarddonner.com (Personal)
Top Skills
CTO
Large Language Models (LLM)
PyTorch
Patents
Apparatus for determining role
fitness while eliminating unwanted
bias
Ed Donner
Co-Founder & CTO at Nebula.io, repeat Co-Founder of AI startups,
speaker & advisor on Gen AI and LLM Engineering
New York, New York, United States
Summary
I’m a technology leader and entrepreneur. I'm applying AI to a field
where it can make a massive impact: helping people discover their
potential and pursue their reason for being. But at my core, I’m a
software engineer and a scientist. I learned how to code aged 8 and
still spend weekends experimenting with Large Language Models
and writing code (rather badly). If you’d like to join us to show me
how it’s done.. message me!
As a work-hobby, I absolutely love giving talks about Gen AI and
LLMs. I'm the author of a best-selling, top-rated Udemy course
on LLM Engineering, and I speak at O'Reilly Live

In [5]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
print(summary)

My name is Ed Donner. I'm an entrepreneur, software engineer and data scientist. I'm originally from London, England, but I moved to NYC in 2000.
I love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.


## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...

__For more, see my companion course AI Engineer Core Track (first week)__

In [7]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [8]:
# FREE-first call: uses Groq Llama 3 70B (or best available FREE provider)
resp = llm_call(messages, want_response_object=True)
print(resp.choices[0].message.content)


  [LLM] Groq FREE Llama 3 70B (model=llama3-70b-8192)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Gemma 2 9B (model=gemma2-9b-it)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Llama 3.1 8B (model=llama-3.1-8b-instant)
  [LLM] failed: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
  [LLM

In [9]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [10]:
resp = llm_call(messages, want_response_object=True)
print(resp.choices[0].message.content)


  [LLM] Groq FREE Llama 3 70B (model=llama3-70b-8192)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Gemma 2 9B (model=gemma2-9b-it)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Llama 3.1 8B (model=llama-3.1-8b-instant)
  [LLM] failed: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
  [LLM

In [11]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [12]:
resp = llm_call(messages, want_response_object=True)
print(resp.choices[0].message.content)


  [LLM] Groq FREE Llama 3 70B (model=llama3-70b-8192)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Gemma 2 9B (model=gemma2-9b-it)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Llama 3.1 8B (model=llama-3.1-8b-instant)
  [LLM] failed: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
  [LLM

In [13]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"},
    {"role": "assistant", "content": "Well hi there, Ed. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [14]:
resp = llm_call(messages, want_response_object=True)
print(resp.choices[0].message.content)


  [LLM] Groq FREE Llama 3 70B (model=llama3-70b-8192)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Gemma 2 9B (model=gemma2-9b-it)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Llama 3.1 8B (model=llama-3.1-8b-instant)
  [LLM] failed: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
  [LLM

## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [15]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [16]:
display(Markdown(system_prompt))



# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

My name is Ed Donner. I'm an entrepreneur, software engineer and data scientist. I'm originally from London, England, but I moved to NYC in 2000.
I love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

   
Contact
ed.donner@gmail.com
www.linkedin.com/in/eddonner
(LinkedIn)
edwarddonner.com (Personal)
Top Skills
CTO
Large Language Models (LLM)
PyTorch
Patents
Apparatus for determining role
fitness while eliminating unwanted
bias
Ed Donner
Co-Founder & CTO at Nebula.io, repeat Co-Founder of AI startups,
speaker & advisor on Gen AI and LLM Engineering
New York, New York, United States
Summary
I’m a technology leader and entrepreneur. I'm applying AI to a field
where it can make a massive impact: helping people discover their
potential and pursue their reason for being. But at my core, I’m a
software engineer and a scientist. I learned how to code aged 8 and
still spend weekends experimenting with Large Language Models
and writing code (rather badly). If you’d like to join us to show me
how it’s done.. message me!
As a work-hobby, I absolutely love giving talks about Gen AI and
LLMs. I'm the author of a best-selling, top-rated Udemy course
on LLM Engineering, and I speak at O'Reilly Live Events and
ODSC workshops. It brings me great joy to help others unlock the
astonishing power of LLMs.
I spent most of my career at JPMorgan building software for financial
markets. I worked in London, Tokyo and New York. I became an MD
running a global organization of 300. Then I left to start my own AI
business, untapt, to solve the problem that had plagued me at JPM -
why is so hard to hire engineers?
At untapt we worked with GQR, one of the world's fastest growing
recruitment firms. We collaborated on a patented invention in AI
and talent. Our skills were perfectly complementary - AI leaders vs
recruitment leaders - so much so, that we decided to join forces. In
2020, untapt was acquired by GQR’s parent company and Nebula
was born.
I’m now Co-Founder and CTO for Nebula, responsible for software
engineering and data science.  Our stack is Python/Flask, React,
Mongo, ElasticSearch, with Kubernetes on GCP. Our 'secret sauce'
is our use of Gen AI and proprietary LLMs. If any of this sounds
interesting - we should talk!
  Page 1 of 5   
Experience
Nebula.io
Co-Founder & CTO
June 2021 - Present (3 years 10 months)
New York, New York, United States
I’m the co-founder and CTO of Nebula.io. We help recruiters source,
understand, engage and manage talent, using Generative AI / proprietary
LLMs. Our patented model matches people with roles with greater accuracy
and speed than previously imaginable — no keywords required.
Our long term goal is to help people discover their potential and pursue their
reason for being, motivated by a concept called Ikigai. We help people find
roles where they will be most fulfilled and successful; as a result, we will raise
the level of human prosperity. It sounds grandiose, but since 77% of people
don’t consider themselves inspired or engaged at work, it’s completely within
our reach.
Simplified.Travel
AI Advisor
February 2025 - Present (2 months)
Simplified Travel is empowering destinations to deliver unforgettable, data-
driven journeys at scale.
I'm giving AI advice to enable highly personalized itinerary solutions for DMOs,
hotels and tourism organizations, enhancing traveler experiences.
GQR Global Markets
Chief Technology Officer
January 2020 - Present (5 years 3 months)
New York, New York, United States
As CTO of parent company Wynden Stark, I'm also responsible for innovation
initiatives at GQR.
Wynden Stark
Chief Technology Officer
January 2020 - Present (5 years 3 months)
New York, New York, United States
With the acquisition of untapt, I transitioned to Chief Technology Officer for the
Wynden Stark Group, responsible for Data Science and Engineering.
  Page 2 of 5   
untapt
6 years 4 months
Founder, CTO
May 2019 - January 2020 (9 months)
Greater New York City Area
I founded untapt in October 2013; emerged from stealth in 2014 and went
into production with first product in 2015. In May 2019, I handed over CEO
responsibilities to Gareth Moody, previously the Chief Revenue Officer, shifting
my focus to the technology and product.
Our core invention is an Artificial Neural Network that uses Deep Learning /
NLP to understand the fit between candidates and roles.
Our SaaS products are used in the Recruitment Industry to connect people
with jobs in a highly scalable way. Our products are also used by Corporations
for internal and external hiring at high volume. We have strong SaaS metrics
and trends, and a growing number of bellwether clients.
Our Deep Learning / NLP models are developed in Python using Google
TensorFlow. Our tech stack is React / Redux and Angular HTML5 front-end
with Python / Flask back-end and MongoDB database. We are deployed on
the Google Cloud Platform using Kubernetes container orchestration.
Interview at NASDAQ: https://www.pscp.tv/w/1mnxeoNrEvZGX
Founder, CEO
October 2013 - May 2019 (5 years 8 months)
Greater New York City Area
I founded untapt in October 2013; emerged from stealth in 2014 and went into
production with first product in 2015.
Our core invention is an Artificial Neural Network that uses Deep Learning /
NLP to understand the fit between candidates and roles.
Our SaaS products are used in the Recruitment Industry to connect people
with jobs in a highly scalable way. Our products are also used by Corporations
for internal and external hiring at high volume. We have strong SaaS metrics
and trends, and a growing number of bellwether clients.
  Page 3 of 5   
Our Deep Learning / NLP models are developed in Python using Google
TensorFlow. Our tech stack is React / Redux and Angular HTML5 front-end
with Python / Flask back-end and MongoDB database. We are deployed on
the Google Cloud Platform using Kubernetes container orchestration.
-- Graduate of FinTech Innovation Lab
-- American Banker Top 20 Company To Watch
-- Voted AWS startup most likely to grow exponentially
-- Forbes contributor
More at https://www.untapt.com
Interview at NASDAQ: https://www.pscp.tv/w/1mnxeoNrEvZGX
In Fast Company: https://www.fastcompany.com/3067339/how-artificial-
intelligence-is-changing-the-way-companies-hire
JPMorgan Chase
11 years 6 months
Managing Director
May 2011 - March 2013 (1 year 11 months)
Head of Technology for the Credit Portfolio Group and Hedge Fund Credit in
the JPMorgan Investment Bank.
Led a team of 300 Java and Python software developers across NY, Houston,
London, Glasgow and India. Responsible for counterparty exposure, CVA
and risk management platforms, including simulation engines in Python that
calculate counterparty credit risk for the firm's Derivatives portfolio.
Managed the electronic trading limits initiative, and the Credit Stress program
which calculates risk information under stressed conditions. Jointly responsible
for Market Data and batch infrastructure across Risk.
Executive Director
January 2007 - May 2011 (4 years 5 months)
From Jan 2008:
Chief Business Technologist for the Credit Portfolio Group and Hedge Fund
Credit in the JPMorgan Investment Bank, building Java and Python solutions
and managing a team of full stack developers.
2007:
  Page 4 of 5   
Responsible for Credit Risk Limits Monitoring infrastructure for Derivatives and
Cash Securities, developed in Java / Javascript / HTML.
VP
July 2004 - December 2006 (2 years 6 months)
Managed Collateral, Netting and Legal documentation technology across
Derivatives, Securities and Traditional Credit Products, including Java, Oracle,
SQL based platforms
VP
October 2001 - June 2004 (2 years 9 months)
Full stack developer, then manager for Java cross-product risk management
system in Credit Markets Technology
Cygnifi
Project Leader
January 2000 - September 2001 (1 year 9 months)
Full stack developer and engineering lead, developing Java and Javascript
platform to risk manage Interest Rate Derivatives at this FInTech startup and
JPMorgan spin-off.
JPMorgan
Associate
July 1997 - December 1999 (2 years 6 months)
Full stack developer for Exotic and Flow Interest Rate Derivatives risk
management system in London, New York and Tokyo
IBM
Software Developer
August 1995 - June 1997 (1 year 11 months)
Java and Smalltalk developer with IBM Global Services; taught IBM classes on
Smalltalk and Object Technology in the UK and around Europe
Education
University of Oxford
Physics  · (1992 - 1995)
  Page 5 of 5

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.


In [17]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [18]:
# First digital twin interaction - automatically uses best FREE provider
answer = llm_call(messages)
display(Markdown(answer))


  [LLM] Groq FREE Llama 3 70B (model=llama3-70b-8192)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Gemma 2 9B (model=gemma2-9b-it)
  [LLM] failed: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
  [LLM] Groq FREE Llama 3.1 8B (model=llama-3.1-8b-instant)
  [LLM] failed: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
  [LLM

Hi there! I'm Ed Donner's digital twin, here to tell you about his journey and expertise.

Ed is an entrepreneur, a software engineer, and a data scientist, originally from London, England, who moved to NYC in 2000. He's currently the Co-Founder and CTO at Nebula.io, where he's applying AI and proprietary Large Language Models to revolutionize talent acquisition, helping people discover their potential and find roles where they'll truly thrive.

Before Nebula, Ed founded untapt, another AI startup focused on connecting people with jobs, which was eventually acquired by GQR’s parent company, leading to the creation of Nebula.io. He also spent a significant part of his career at JPMorgan Chase, building software for financial markets and ultimately leading a global organization of 300 developers.

Ed is deeply passionate about Generative AI and LLM Engineering. He enjoys giving talks on these subjects, is the author of a best-selling Udemy course, and speaks at events like O'Reilly Live and ODSC workshops. At his core, he's a software engineer who loves to experiment with LLMs and code in his free time.

On a more personal note, Ed is a big fan of all foods, especially French cuisine. He has a unique aversion to most cheeses, though he makes exceptions for cream cheese and mozzarella – so cheesecake and pizza are definitely on his favorites list!

What would you like to know more about Ed's work or experience?

In [19]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    answer = llm_call(messages, verbose=False)
    # llm_call returns the content string by default — perfect for Gradio ChatInterface
    return answer


In [20]:
chat("Please summarize who you are", [])

"Hello there! I'm Ed Donner's digital twin, an AI designed to represent him and answer questions about his professional background, skills, and experience.\n\nEd is an entrepreneur, software engineer, and data scientist with a passion for applying AI to help people discover their potential and pursue fulfilling careers. He's currently the Co-Founder and CTO of Nebula.io, a company that uses Generative AI and proprietary LLMs to help recruiters and talent find the right matches.\n\nThroughout his career, Ed has been a repeat co-founder of AI startups. Before Nebula.io, he founded untapt, another AI business focused on solving hiring challenges. He also spent a significant part of his career at JPMorgan Chase, where he rose to become a Managing Director, leading a global organization of 300 software developers.\n\nHe's also a speaker and advisor on Generative AI and LLM Engineering, and has even authored a best-selling Udemy course on the topic. At his core, he's a software engineer who 

## NOTE for those not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\nagal\.agents\AGENTS\AGENTS\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\nagal\.agents\AGENTS\AGENTS\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


# And now - TOOLS!

Let's start with a function...

In [22]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [23]:
record_email_tool("test@testy.com")

Tool called to record an email: test@testy.com


'Email received'

## Step 1 - write some json to describe the tool


In [24]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [25]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [26]:
tools

[{'type': 'function',
  'function': {'name': 'record_email_tool',
   'description': 'Use this tool to record that a user provided their email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'}},
    'required': ['email'],
    'additionalProperties': False}}}]

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [27]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    # 1. First LLM call — passes tools (may return tool_calls)
    response = llm_call(messages, tools=tools, want_response_object=True)

    choice = response.choices[0]
    message = choice.message

    # 2. If the LLM wants to call a tool: execute it, append result, call LLM again
    if getattr(message, "tool_calls", None):
        # Extend conversation with the assistant's tool-request message
        messages.append(message)

        for tool_call in message.tool_calls:
            tool_name  = tool_call.function.name
            tool_args  = json.loads(tool_call.function.arguments)

            print("The LLM wants to call tool: {} with args {}".format(tool_name, tool_args))

            # Dispatch to the actual Python function
            if tool_name == "record_email_tool":
                tool_result = record_email_tool(**tool_args)
            else:
                tool_result = "Unknown tool: {}".format(tool_name)

            # Append the tool response back to the conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(tool_result),
            })

        # 3. Second LLM call — now with tool results, produces final answer
        second_response = llm_call(messages, want_response_object=True)
        return second_response.choices[0].message.content

    # No tool call needed - just return the content
    return message.content


In [28]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [29]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    # Agent loop: up to 5 iterations of think → tool → think → ... → final answer
    for _ in range(5):
        response = llm_call(messages, tools=tools, want_response_object=True)
        choice = response.choices[0]
        message = choice.message
        messages.append(message)

        if getattr(message, "tool_calls", None):
            # Run each tool the LLM requested
            for tool_call in message.tool_calls:
                tool_name  = tool_call.function.name
                tool_args  = json.loads(tool_call.function.arguments)

                print("Agent loop tool call: {} with args {}".format(tool_name, tool_args))

                if tool_name == "record_email_tool":
                    tool_result = record_email_tool(**tool_args)
                else:
                    tool_result = "Unknown tool: {}".format(tool_name)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(tool_result),
                })
            # Loop back for another LLM turn with tool results in the conversation
        else:
            # No more tool calls → final answer
            return message.content

    # Fallback (if we exhausted the 5 turn limit without a non-tool answer)
    return "Sorry - I wasn't able to complete the task in the allowed steps."


In [30]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>
        </td>
    </tr>
</table>